# Lecția 08 - Modelul de Proiectare Multi-Agent


## Configurare


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## De ce Sisteme Multi-Agent?

Sarcinile din lumea reală, precum planificarea unei călătorii, implică multe tipuri diferite de expertiză — logistică, cunoștințe locale, bugetare și altele. Un singur agent care încearcă să gestioneze totul devine rapid greu de controlat.

Sistemele multi-agent rezolvă această problemă prin **specializare**: fiecare agent se concentrează pe un domeniu de expertiză, generând rezultate de o calitate superioară față de un generalist. De asemenea, ele îmbunătățesc **scalabilitatea** — poți adăuga agenți noi (de exemplu, un specialist în zboruri, un critic de restaurante) fără a rescrie fluxul de lucru existent. Agenții se compun între ei printr-un pipeline structurat, transferând contextul de la unul la altul.


## Crearea agenților specializați


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Construirea unui flux de lucru secvențial

`WorkflowBuilder` îți permite să conectezi agenți într-un graf direcționat. Aici creăm o conductă simplă în doi pași: **TravelPlanner** elaborează itinerariul, apoi **TravelConcierge** îl revizuiește și îl îmbunătățește.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Adăugarea mai multor agenți în fluxul de lucru

Unul dintre cele mai mari avantaje ale modelului multi-agent este cât de ușor este de extins. Mai jos adăugăm un agent **BudgetReviewer** care verifică planul în raport cu bugetul călătorului, marchează articolele care ar putea depăși limita costurilor și sugerează alternative economice. Fluxul de lucru rulează acum trei agenți în secvență:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Rezumat

În această lecție ai învățat cum să:

1. **Creezi agenți specializați** — fiecare cu un rol concentrat (planificare, concierge, revizuirea bugetului).
2. **Conectezi agenții într-un flux de lucru secvențial** folosind `WorkflowBuilder` și `add_edge`.
3. **Transmiți fluxul de ieșire** dintr-un pipeline multi-agent, urmărind care agent vorbește.
4. **Extinzi un flux de lucru** adăugând noi agenți în lanț fără a modifica agenții existenți.

Modelul de design multi-agent păstrează fiecare agent simplu, în timp ce produce rezultate mai bogate, mai bine revizuite decât ar putea obține un singur agent de unul singur.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
